# Voxtral-Mini-3B on AWS Neuron (trn2.3xlarge)

This notebook demonstrates how to download, compile, and serve
**Voxtral-Mini-3B** (`mistralai/Voxtral-Mini-3B-2507`) on `trn2.3xlarge`
using **vLLM-neuron** with the NxDI (`neuronx-distributed-inference`)
Voxtral contrib.

## Instance Setup

This notebook is designed to run on a **trn2.3xlarge** with the
**Deep Learning AMI Neuron (Ubuntu 24.04) 20260522** (Neuron SDK 2.30).

To launch your instance, use the AWS Console or CLI.  For a walkthrough
of launching a Neuron instance, see this video tutorial (starts at the
instance launch section):

> **Video Guide**: [Launching a Neuron Instance](https://youtu.be/CyTCTuq1z0Q?t=657)

## Jupyter Kernel Setup

This notebook requires a Python kernel from the pre-installed Neuron
virtual environment.  There are two ways to set this up:

### Option A: Jupyter Server on the Instance

SSH into your instance and start Jupyter from the Neuron virtual
environment:

```bash
source /opt/aws_neuronx_venv_pytorch_inference_vllm_0_16/bin/activate
pip install jupyter
jupyter notebook --no-browser --port=8888
```

Then use SSH port forwarding to access it from your local browser:

```bash
ssh -i "/path/to/sshkey.pem" -L 8888:localhost:8888 ubuntu@<instance_ip>
```

### Option B: VS Code Remote-SSH

With Visual Studio Code installed on your local machine, you can use
Remote-SSH to edit and run notebooks directly on the Neuron instance:

1. Select **Remote-SSH: Connect to Host...** from the Command Palette
   (`F1` or `Shift+Cmd+P`)
2. Enter the full connection string:
   `ssh -i "/path/to/sshkey.pem" ubuntu@<instance_ip>`
3. VS Code will connect and set up the VS Code server automatically
4. When prompted, browse to your working directory on the instance
5. Some menu commands may appear greyed out, but keyboard shortcuts
   still work (`Cmd+S` to save, `` Ctrl+Shift+` `` for terminal).  You
   may need to restart VS Code.

To use the pre-installed Neuron virtual environment as your Jupyter
kernel in VS Code, open a terminal and create a symbolic link:

```bash
ln -s /opt/aws_neuronx_venv_pytorch_inference_vllm_0_16 ~/.venv
```

Then select the `.venv` Python interpreter when choosing a kernel for
the notebook.

## What this notebook does

1. Clone the `jimburtoft` forks of `neuronx-distributed-inference`
   (branch `contrib/voxtral-mini-3B`) and `vllm-neuron` (branch
   `contrib/voxtral-0.5.0`).  These add Voxtral support that is not yet
   merged into the upstream projects.
2. Install both forks in the pre-installed SDK 2.30 vLLM venv.
3. Download the Voxtral-Mini-3B checkpoint from HuggingFace.
4. Compile the model to Neuron (one-time; a few minutes).
5. Run a single-file smoke transcription via vLLM.
6. Run the benchmark harness across your own audio dataset and report
   per-file mean latency.

The performance target is **mean ≤ 543 ms per file** on a mix of 0-30 s
speech clips at TP=4, LNC=2, bfloat16, SDK 2.30.


## 1. Verify the instance

Run `neuron-ls` and check that you have four logical NeuronCores
(LNC=2 default on trn2.3xlarge).


In [ ]:
!neuron-ls


## 2. Activate the pre-installed venv

The SDK 2.30 DLAMI ships two venvs relevant to this notebook:

- `/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/` -- NxDI only.
- `/opt/aws_neuronx_venv_pytorch_inference_vllm_0_16/` -- vLLM + NxDI.
  **This is the one we want.**

We assume your Jupyter kernel already comes from the vLLM venv (see the
"Jupyter Kernel Setup" section at the top of the notebook).  Verify
that the runtime venv is correct:


In [ ]:
import sys
print("Python:", sys.executable)
assert "aws_neuronx_venv_pytorch_inference_vllm_0_16" in sys.executable, (
    "This notebook expects the pre-installed vLLM venv.  See kernel setup."
)


## 3. Clone the forks

Both the Voxtral NxDI implementation and the vLLM-neuron Voxtral
integration live on `jimburtoft`-hosted forks.  Clone them into your
home directory.  The NxDI branch will be added to `PYTHONPATH` at
runtime; the vLLM-neuron branch is installed via `pip install -e`.


In [ ]:
import os
from pathlib import Path

HOME = Path(os.path.expanduser("~"))
NXDI_FORK = HOME / "neuronx-distributed-inference"
VLLM_FORK = HOME / "vllm-neuron"

if not NXDI_FORK.exists():
    !git clone -b contrib/voxtral-mini-3B \
        https://github.com/jimburtoft/neuronx-distributed-inference.git \
        {NXDI_FORK}
else:
    print(f"{NXDI_FORK} already exists, skipping clone.")

if not VLLM_FORK.exists():
    !git clone -b contrib/voxtral-0.5.0 \
        https://github.com/jimburtoft/vllm-neuron.git \
        {VLLM_FORK}
else:
    print(f"{VLLM_FORK} already exists, skipping clone.")


## 4. Install the vLLM-neuron fork

This replaces the DLAMI's pre-installed `vllm-neuron` package with the
patched fork.  The install is editable (`-e`) so future `git pull`
updates take effect without a reinstall.


In [ ]:
!pip install -e {VLLM_FORK} 2>&1 | tail -5


Verify the install picked up the Voxtral-patched loader:


In [ ]:
import importlib
import vllm_neuron.worker.neuronx_distributed_model_loader as loader
importlib.reload(loader)
assert hasattr(loader, "NeuronVoxtralForCausalLM"), (
    "vllm-neuron fork Voxtral class missing"
)
print("NeuronVoxtralForCausalLM present.")


## 5. Install audio dependencies

`mistral_common[audio]` provides the tokenizer and audio pre-processor
that Voxtral's chat template uses.  `soundfile` and `librosa` are for
the benchmark harness's audio loader.  (Most of these are already in
the DLAMI venv; the cell is a no-op if so.)


In [ ]:
!pip install --quiet 'mistral_common[audio]>=1.8.1' 'transformers>=4.54.0' \
    soundfile librosa 2>&1 | tail -3


## 6. Download the Voxtral-Mini-3B checkpoint

Voxtral-Mini-3B is a **gated** repo on HuggingFace.  Before running the
next cell:

1. Visit <https://huggingface.co/mistralai/Voxtral-Mini-3B-2507> and
   accept the license.
2. Generate an HF access token at
   <https://huggingface.co/settings/tokens>.
3. Run `huggingface-cli login` in a terminal and paste your token, OR
   set `HF_TOKEN` in the cell below.

The download is ~10 GB.


In [ ]:
MODEL_DIR = HOME / "models" / "Voxtral-Mini-3B-2507"

if not MODEL_DIR.exists():
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id="mistralai/Voxtral-Mini-3B-2507",
        local_dir=str(MODEL_DIR),
        # token=os.environ.get("HF_TOKEN"),  # or run `huggingface-cli login`
    )
else:
    print(f"{MODEL_DIR} already exists.")
!ls -lah {MODEL_DIR} | head


## 7. Point vLLM at the NxDI Voxtral contrib

vLLM-neuron's Voxtral loader imports `NeuronApplicationVoxtral` from
the NxDI contrib.  Add the contrib `src/` to `PYTHONPATH` so the
vLLM engine's worker process can import it.


In [ ]:
CONTRIB_SRC = NXDI_FORK / "contrib" / "models" / "voxtral-mini-3B" / "src"
sys.path.insert(0, str(CONTRIB_SRC))
os.environ["PYTHONPATH"] = (
    str(CONTRIB_SRC) + os.pathsep + os.environ.get("PYTHONPATH", "")
)
print("Contrib src:", CONTRIB_SRC)

from modeling_voxtral import NeuronApplicationVoxtral  # noqa: F401
print("Import OK.")


## 8. Compile the model (one-time, a few minutes)

vLLM will compile Voxtral on first use.  To make the walkthrough
deterministic, we compile ahead of time via
`NeuronApplicationVoxtral.compile()`.  The flags we pick -- TP=4,
`n_positions=768`, `seq_len=768`, on-device sampling, and
`move_trace_to_device` -- match the recommended production
configuration documented in the parent README (§ "Optimizations
shipped in this contrib").

**Important**: `seq_len=768` matches `n_positions=768`.  When serving
via vLLM V1's continuous scheduler this constraint prevents the NxDI
TKG NEFF from being fed positions outside its compiled range during
decode.  Standalone `NeuronApplicationVoxtral.transcribe()` works
with any `seq_len < n_positions`, but vLLM 0.16.0 requires them to
match.

Subsequent notebook runs skip compilation and reload from
`COMPILED_DIR`.


In [ ]:
import torch
COMPILED_DIR = HOME / "compiled" / "voxtral_mini_3b_tp4_ods_768_768"
os.environ["NEURON_COMPILED_ARTIFACTS"] = str(COMPILED_DIR)

marker = COMPILED_DIR / "text_decoder" / "text_model" / "model.pt"
if marker.exists():
    print(f"Already compiled at {COMPILED_DIR}.")
else:
    print(f"Compiling to {COMPILED_DIR}. This takes several minutes"
          " the first time.")
    app = NeuronApplicationVoxtral(
        model_path=str(MODEL_DIR),
        tp_degree=4,
        seq_len=768,             # matches n_positions
        n_positions=768,         # KV cache sized for a single 30 s clip
        dtype=torch.bfloat16,
        on_device_sampling=True,     # greedy argmax on-device
        move_trace_to_device=True,   # pre-stage encoder on NeuronCore 0
    )
    app.compile(str(COMPILED_DIR))
    del app
    import gc; gc.collect()
    print("Compile done.")


## 9. Launch vLLM

vLLM-neuron loads the pre-compiled model from
`NEURON_COMPILED_ARTIFACTS` and serves it via the standard `LLM`
in-process API.  We keep the engine in-process (rather than the OpenAI
HTTP server) so this notebook is self-contained.

**`max_num_seqs=1` is required.**  Batch size > 1 is not supported by
this contrib (see Known Limitations at the end of the notebook).

**`allowed_local_media_path='/'`** lets vLLM read audio files from
anywhere on the filesystem.  Restrict this to your dataset directory
in production.


In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(
    model=str(MODEL_DIR),
    tokenizer_mode="mistral",
    tensor_parallel_size=4,      # TP=4 on trn2.3xlarge (LNC=2, four cores)
    dtype="bfloat16",
    max_num_seqs=1,              # BS>1 not supported
    max_model_len=768,           # must match compiled n_positions
    enable_prefix_caching=False,
    allowed_local_media_path="/",
    additional_config={
        "override_neuron_config": {
            "on_device_sampling": True,
            "move_trace_to_device": True,
            "n_positions": 768,
            "seq_len": 768,
        },
    },
)
print("vLLM engine ready.")


## 10. Single-file smoke test

Transcribe one 15 s TED-style clip to prove the pipeline works.
vLLM's Mistral chat template accepts audio inputs via
`{"type": "audio_url", "audio_url": {"url": <path>}}`.

Provide any 16 kHz mono `.wav` file (or a URL that `mistral_common`
can download).  The cell below fetches a public TED sample and
trims/downsamples it to 15 s mono 16 kHz.


In [ ]:
import urllib.request
AUDIO_URL = (
    "https://huggingface.co/datasets/reach-vb/random-audios/"
    "resolve/main/ted_60.wav"
)
RAW_AUDIO = HOME / "sample_audio" / "ted_60.wav"
SAMPLE_AUDIO = HOME / "sample_audio" / "ted_15_mono16k.wav"
RAW_AUDIO.parent.mkdir(parents=True, exist_ok=True)

if not RAW_AUDIO.exists():
    urllib.request.urlretrieve(AUDIO_URL, RAW_AUDIO)

if not SAMPLE_AUDIO.exists():
    import soundfile as sf, librosa, numpy as np
    data, sr = sf.read(str(RAW_AUDIO))
    if data.ndim == 2:
        data = data.mean(axis=1)          # stereo -> mono
    data = librosa.resample(data.astype(np.float32),
                            orig_sr=sr, target_sr=16000)
    data = data[:15 * 16000]              # trim to 15 s
    sf.write(str(SAMPLE_AUDIO), data, 16000)

print("Audio file:", SAMPLE_AUDIO,
      SAMPLE_AUDIO.stat().st_size, "bytes")


In [ ]:
conversation = [{
    "role": "user",
    "content": [
        {"type": "audio_url",
         "audio_url": {"url": f"file://{SAMPLE_AUDIO}"}},
        {"type": "text", "text": "Transcribe this audio."},
    ],
}]

sampling = SamplingParams(temperature=0.0, max_tokens=256)

import time
t0 = time.perf_counter()
outputs = llm.chat(conversation, sampling_params=sampling)
elapsed = time.perf_counter() - t0

text = outputs[0].outputs[0].text
print(f"Latency: {elapsed * 1000:.1f} ms")
print(f"Tokens:  {len(outputs[0].outputs[0].token_ids)}")
print(f"\nTranscription:\n{text}")


## 11. Run the benchmark harness

The `benchmark_harness/` directory alongside this notebook has a
serial driver that measures latency per audio file.  To run it:

1. Populate `benchmark_harness/dataset/` with your own audio files
   (16 kHz mono WAV recommended).
2. Add one row per file to `benchmark_harness/dataset/manifest.csv`
   in the `audio_path,duration_sec,transcript` format.
3. Run the cell below.

**Reference measurement**: 5 clips of 5/10/15/20/25 seconds from
`reach-vb/random-audios/ted_60.wav`, 3 runs, `max_new_tokens=256`.

| Duration | Mean per file (ms) |
|:--------:|:------------------:|
| 5-10 s   | ~291               |
| 10-15 s  | ~408               |
| 15-20 s  | ~454               |
| 20-25 s  | ~626               |
| 25-30 s  | ~733               |
| Overall  | **~502 ms mean, 456 ms median** |

**Your target is ≤ 543 ms mean per file.**  Your numbers will vary
with content complexity (dense speech → more generated tokens →
longer per-file latency).

The harness uses `NeuronApplicationVoxtral` under the hood via the
`vllm_neuron` backend, so the vLLM engine and the harness measure
the same code path.


In [ ]:
HARNESS = NXDI_FORK / "contrib" / "models" / "voxtral-mini-3B" / \
          "benchmark_harness"
MANIFEST = HARNESS / "dataset" / "manifest.csv"

# Count how many rows the manifest has (header + data lines)
n_manifest = sum(1 for _ in MANIFEST.open()) - 1
print(f"Manifest at {MANIFEST}: {n_manifest} audio rows.")

if n_manifest == 0:
    print()
    print("NOTE: The manifest is empty.  Populate it before running")
    print("      the cell below.  See benchmark_harness/dataset/README.md.")


In [ ]:
import subprocess

if n_manifest > 0:
    # Free the in-process vLLM engine before the harness runs -- the
    # harness will re-create it internally and only one process at a
    # time can hold the Neuron cores.
    try:
        del llm
    except NameError:
        pass
    import gc; gc.collect()

    cmd = [
        "python", str(HARNESS / "run_voxtral_benchmark.py"),
        "--backend", "vllm_neuron",
        "--manifest", str(MANIFEST),
        "--model-dir", str(MODEL_DIR),
        "--compiled-dir", str(COMPILED_DIR),
        "--tp-degree", "4",
        "--seq-len", "768",
        "--n-positions", "768",
        "--ods",
        "--output-dir", str(HARNESS / "results"),
        "--runs", "3",
    ]
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=False)
    print("Exit code:", result.returncode)
else:
    print("Skipping -- manifest is empty.")


In [ ]:
if n_manifest > 0:
    !python {HARNESS}/summarize_results.py {HARNESS}/results/
else:
    print("Skipping -- no results to summarize.")


## 12. Known Limitations

- **Batch size 1 only.**  The stock `ImageToTextModelWrapper` uses
  `scatter_by_index_put` in a way that is not shape-safe for batch
  size > 1 context encoding.  `max_num_seqs=1` is set above.
  Serving concurrent requests requires multiple engines (one per
  NeuronCore group) or an unmerged scatter fix.
- **30 s audio maximum per request.**  Voxtral upstream supports 30
  min transcribe / 40 min understand modes; those code paths have not
  been validated on Neuron.  Clips longer than 30 s are truncated by
  the audio encoder.
- **`seq_len` must match `n_positions` for vLLM serving.**  Standalone
  `NeuronApplicationVoxtral.transcribe()` works with `seq_len` shorter
  than `n_positions` (e.g. 512 / 768) which is ~5% faster on SDK 2.30.
  vLLM 0.16.0's V1 scheduler can feed the TKG NEFF positions between
  `seq_len` and `n_positions`, which the compiled NEFF rejects with
  NRT status 1006.  Keep `seq_len == n_positions` for vLLM.
- **Continuous batching and streaming responses** (`stream=true` on
  the HTTP API) have not been benchmarked on this configuration.
- **Function calling** (Voxtral-Small-24B only) is out of scope for
  this notebook.
- **Voxtral-Small-24B** is not covered; it uses the same architecture
  and could be onboarded at TP=4 with the same
  `NeuronApplicationVoxtral` pattern but has not been validated in
  this branch.

## Where to file issues

- vLLM-neuron Voxtral loader:
  <https://github.com/jimburtoft/vllm-neuron/tree/contrib/voxtral-0.5.0>
- NxDI Voxtral contrib:
  <https://github.com/jimburtoft/neuronx-distributed-inference/tree/contrib/voxtral-mini-3B>

After the upstream PR merges, the branches above will be replaced by
`aws-neuron/neuronx-distributed-inference` and
`vllm-project/vllm-neuron` releases.
